In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from gridworld_env_assignment import GridWorld
from utils import paint_gridworld

## Checking the initial environments.
Construct a `GridWorld` with appropriate input arguments for each of the two tasks. Use `paint_gridworld` to plot the resulting environments. 

In [ ]:
example_env = GridWorld(goal_locations=[(1,1)], goal_rewards=[100])

paint_gridworld(example_env)

## Example of running a simulation 

In [ ]:
# Section 3: Learning to navigate with Q-Learning

# Hyperparameters
eta = 0.1   # learning rate
gamma = 0.99  # discount factor
epsilon = 0.3 # exploration rate
num_episodes = 10000

# Initialize environment
example_env = GridWorld(goal_locations=[(2,1), (9,8)], goal_rewards=[250, 5000])

# Initialize Q-table
Q = np.zeros((example_env.get_state_size(), example_env.get_action_size()))

# Storage for training curves
reward_per_episode = []
steps_per_episode = []

# Training loop
for episode in range(num_episodes):
    _, state, reward, done = example_env.reset()
    total_reward = 0
    steps = 0

    while not done:
        # ε-greedy action selection
        if np.random.rand() < epsilon:
            action = np.random.choice(example_env.get_action_size())
        else:
            action = np.argmax(Q[state])

        # Take action
        _, next_state, reward, done = example_env.step(action)

        # Q-learning update
        best_next_q = np.max(Q[next_state])
        Q[state, action] += eta * (reward + gamma * best_next_q - Q[state, action])

        state = next_state
        total_reward += reward
        steps += 1

    reward_per_episode.append(total_reward)
    steps_per_episode.append(steps)


# Create an RL Agent

Implement either SARSA or Q-learning to determine how your agent executes actions and learns from the gathered rewards.

In [ ]:
def learn_policy(env: GridWorld, n_episodes: int = 1) -> tuple[np.ndarray, list[float], list[int]]:
    

    # Hyperparameters
    eta = 0.1
    gamma = 0.99
    epsilon = 0.3

    # Initialize Q-table
    Q = np.zeros((env.get_state_size(), env.get_action_size()))

    total_rewards = []
    episode_lengths = []

    for episode in range(n_episodes):
        _, state, reward, done = env.reset()
        total_reward = 0
        steps = 0

        while not done:
            # ε-greedy action selection
            if np.random.rand() < epsilon:
                action = np.random.choice(env.get_action_size())
            else:
                action = np.argmax(Q[state])

            # Take action
            _, next_state, reward, done = env.step(action)

            # Q-learning update
            best_next_q = np.max(Q[next_state])
            Q[state, action] += eta * (reward + gamma * best_next_q - Q[state, action])

            state = next_state
            total_reward += reward
            steps += 1

        total_rewards.append(total_reward)
        episode_lengths.append(steps)

    # Extract final policy
    policy = np.argmax(Q, axis=1)

    return policy, total_rewards, episode_lengths


# Learn Policy for Dual Targets with Differing Rewards
Learn a policy for an environment with two goals, where the more distant goal is considerably more valuable. 

TIP: Test the learning algorithm with an environment that has just a single reward, and in different positions. 

TIP: Modify the relative rewards between the two goals to see changes in behavior. 

In [ ]:
##section 4.2 Learning Curves
env = GridWorld(goal_locations=[(2,1), (9,8)], goal_rewards=[250, 5000])
paint_gridworld(env)


policy, total_rewards, total_steps = learn_policy(env, n_episodes=10000)



# Convert to NumPy arrays
reward_array = np.array(total_rewards)
steps_array = np.array(total_steps)

# Compute moving average over every 10 episodes
window = 10
avg_rewards = reward_array.reshape(-1, window).mean(axis=1)
avg_steps = steps_array.reshape(-1, window).mean(axis=1)

# Plot averaged total reward
plt.plot(avg_rewards)
plt.xlabel('Episodes (grouped in tens)')
plt.ylabel('Average Total Reward')
plt.title('Q-Learning: Smoothed Reward Curve')
plt.grid(True)
plt.show()

# Plot averaged steps per episode
plt.plot(avg_steps)
plt.xlabel('Episodes (grouped in tens)')
plt.ylabel('Average Steps Taken')
plt.title('Q-Learning: Smoothed Steps per Episode')
plt.grid(True)
plt.show()


In [ ]:
# Do your plots below! Include any new function definitions in this jupyter notebook!
##Section 4.4: Policy Visualisation

grid_shape = env.get_gridshape()
walls = env.get_walls_loc()
cliffs = env.get_cliffs_loc()
goals = env.get_goal_loc()

fig, ax = plt.subplots(figsize=(7, 7))

# Set axis limits
ax.set_xlim(0, grid_shape[1])
ax.set_ylim(0, grid_shape[0])

# Draw walls
for (i, j) in walls:
    ax.add_patch(plt.Rectangle((j, grid_shape[0] - i - 1), 1, 1, color='black'))

# Draw cliffs
for (i, j) in cliffs:
    ax.add_patch(plt.Rectangle((j, grid_shape[0] - i - 1), 1, 1, color='dodgerblue'))

# Draw goals
for (i, j) in goals:
    ax.add_patch(plt.Rectangle((j, grid_shape[0] - i - 1), 1, 1, color='firebrick'))

# Plot best action at each state (from policy)
for state in range(env.get_state_size()):
    loc = env.get_loc_from_state(state)
    i, j = loc
    best_action = policy[state]

    dx, dy = 0, 0
    if best_action == 0:  # North
        dy = -0.4
    elif best_action == 1:  # East
        dx = 0.4
    elif best_action == 2:  # South
        dy = 0.4
    elif best_action == 3:  # West
        dx = -0.4

    if (i, j) not in walls:
        ax.arrow(j + 0.5, grid_shape[0] - i - 0.5, dx, dy,
                 head_width=0.2, head_length=0.2, fc='k', ec='k')

# Set ticks at the center of each cell
ax.set_xticks(np.arange(0.5, grid_shape[1], 1))
ax.set_yticks(np.arange(0.5, grid_shape[0], 1))
ax.set_xticklabels(np.arange(0, grid_shape[1]))
ax.set_yticklabels(np.arange(0, grid_shape[0]))
ax.tick_params(axis='both', which='both', length=0)

# Draw grid lines
for x in range(grid_shape[1] + 1):
    ax.axvline(x, color='black', linewidth=1)
for y in range(grid_shape[0] + 1):
    ax.axhline(y, color='black', linewidth=1)

ax.set_aspect('equal')
ax.invert_yaxis()
plt.show()
